# Grouped-Neighborhood Linear Regression (GNL) Imputation

This notebook imputes missing neutralization values in an antibody–virus matrix
using **Grouped Linear Regression (GLR)** based on viral envelope sequences.

It supports two modes:
1. **Genuine missing values** — imputes entries that were never measured.
2. **Withheld values** (for evaluation) — randomly withholds observed entries and evaluates imputation accuracy.


## Prerequisites

> **Important:** Before running this notebook, you must preprocess the raw data by running:
> 1. `process.ipynb` — generates filtered neutralization matrices and alignment-based PCA features.
> 2. `process_non-alignment-features.ipynb` — generates alignment-free features.
>
> Both notebooks write their outputs to `../processed-data/`, which this notebook reads.


## Usage

1. Edit the parameters in the **Configuration** cell below.
2. Run all cells in order (Kernel → Restart & Run All).
3. Results are written to `../output/`.

**Key output files:**
| File | Description |
|------|-------------|
| `Imputed_mat_GLR-{key}_{type}.txt` | Full imputed matrix (GLR) |
| `imputed_ic50-{key}_{type}.csv` | Withheld entries with true vs. imputed values |
| `bool_mat_NaN-Observed-Witheld-{key}_{type}.txt` | O/N/W status matrix |
| `rank_dependency_id-{key}_{type}.txt` | Rank vs. accuracy table |
| `rank_dependency_optimal_id-{key}_{type}.txt` | Accuracy at optimal rank |


In [5]:
using DelimitedFiles, StatsBase, Random, Statistics, Distributions, LinearAlgebra, Clustering, Printf, CSV, DataFrames

# Load helper functions for imputation (see ../src/tools_imputation_minimum.jl)
DIR_SOURCE_CODE = "../src"
include("$(DIR_SOURCE_CODE)/tools_imputation_minimum.jl");

## Configuration

Modify the parameters below before running the simulation.
All other cells should run without modification.


In [6]:
# ============================================================
# Directories (relative to this notebook's location in note/)
# ============================================================
DIR_IN  = "../processed-data"   # preprocessed data (output of process.ipynb)
DIR_OUT = "../output"           # where results are written
DIR_RAW = "../raw-data"         # raw input files (e.g., antibody class table)

# ============================================================
# Experiment identifier
# ============================================================
file_key  = "test"    # label appended to all output filenames

# ============================================================
# Data type to impute
# ============================================================
DATA_TYPE = "IC50"  # options: "IC50" | "IC80" | "HillCoeff" | "IIP"

# ============================================================
# Masking / withholding scheme (for cross-validation)
# ============================================================
# 1: random
# 2: withhold from antibodies with many observations
# 3: withhold from antibodies with few observations
# 4: more_observed (unbiased)
# 5: less_observed (unbiased)
# 6: withhold viruses with high sequence similarity
# 7: withhold viruses with low sequence similarity
# 8: withhold never-observed viruses with high similarity
# 9: withhold never-observed viruses with low similarity
types_of_masking = 1

ratio_withold = 0.1   # fraction of observed values to withhold (e.g., 0.1 = 10%)

# ============================================================
# Minimum data requirements
# ============================================================
num_minimum_num_ic50_each_abs = 1   # min IC50 measurements per antibody
num_minimum_num_ic50_each_vrs = 10  # min IC50 measurements per virus

# ============================================================
# GLR model hyperparameters
# ============================================================
λ_reg           = 10     # L2 regularization for single-antibody groups
λ1_eff          = λ_reg  # (derived) per-antibody regularization strength
λ2_eff          = 5      # cross-antibody Laplacian smoothness regularization
n_effective_dim = 100    # number of PCA/feature dimensions for virus representation
k_max           = 100    # number of antibody clusters (k-means)
n_max_eigen     = Int(floor(0.5 * k_max))  # eigenvectors used for spectral clustering
d_scale         = 0.5    # effective regression dimension = d_scale × #observed viruses
d_eff_max       = 100    # hard cap on effective regression dimension

# ============================================================
# Algorithm and output flags
# ============================================================
flag_subtract_mean    = false  # subtract per-antibody mean before imputation
flag_eff_GLR          = false  # use efficient (Woodbury) GLR solver [experimental]
flag_param_out_GLR    = true   # export GLR parameters to CSV
flag_similarities_out = flag_param_out_GLR  # export antibody cluster assignments
flag_out_imputation   = true   # export imputed values to CSV

# ============================================================
# Ranks to evaluate in the rank-dependency analysis
# ============================================================
rank_set = [10, 20, 30, 40, 50, 60, 70, 80, 90, 100, 200, 300, 400]

# --- Log all conditions to file ---
fout = open(@sprintf("%s/log-%s.txt", DIR_OUT, file_key), "w")
println(fout, @sprintf("######### CONDITIONS #########\ndir_in=%s\ndir_out=%s\ntypes_of_masking=%d\nratio_withold=%f\nflag_param_out_GLR=%s\nflag_out_imputation=%s\nλ_reg=%f\nλ2_eff=%f\nn_effective_dim=%d\nfile_key=%s\nk_max=%d\nflag_similarities_out=%s",
    DIR_IN, DIR_OUT, types_of_masking, ratio_withold, string(flag_param_out_GLR), string(flag_out_imputation), λ_reg, λ2_eff, n_effective_dim, file_key, k_max, string(flag_similarities_out)))
println(@sprintf("######### CONDITIONS #########\ndir_in=%s\ndir_out=%s\ntypes_of_masking=%d\nratio_withold=%f\nflag_param_out_GLR=%s\nflag_out_imputation=%s\nλ_reg=%f\nλ2_eff=%f\nn_effective_dim=%d\nfile_key=%s\nk_max=%d\nflag_similarities_out=%s",
    DIR_IN, DIR_OUT, types_of_masking, ratio_withold, string(flag_param_out_GLR), string(flag_out_imputation), λ_reg, λ2_eff, n_effective_dim, file_key, k_max, string(flag_similarities_out)))
println(@sprintf("num_minimum_num_ic50_each_abs=%d\nnum_minimum_num_ic50_each_vrs=%d", num_minimum_num_ic50_each_abs, num_minimum_num_ic50_each_vrs))
println(fout, @sprintf("num_minimum_num_ic50_each_abs=%d\nnum_minimum_num_ic50_each_vrs=%d", num_minimum_num_ic50_each_abs, num_minimum_num_ic50_each_vrs))
close(fout);

######### CONDITIONS #########
dir_in=../processed-data
dir_out=../output
types_of_masking=1
ratio_withold=0.100000
flag_param_out_GLR=true
flag_out_imputation=true
λ_reg=10.000000
λ2_eff=5.000000
n_effective_dim=100
file_key=test
k_max=100
flag_similarities_out=true
num_minimum_num_ic50_each_abs=1
num_minimum_num_ic50_each_vrs=10


## Step 1: Load Data

Load the neutralization matrix and sequence features from preprocessed files.


In [7]:
# Note: reducing n_max_eigen gives more balanced clusters but may reduce accuracy.
# Higher eigenvalues are necessary for correct classification.
fname_abs_class = "$(DIR_RAW)/catnap_antibody-types_w_class_O.csv"

IC50_single_w_seq = if DATA_TYPE == "IC50"
    readdlm("$(DIR_OUT)/IC50_single_w_seq_O.txt")
elseif DATA_TYPE == "IC80"
    readdlm("$(DIR_OUT)/IC80_single_w_seq_O.txt")
elseif DATA_TYPE == "HillCoeff"
    readdlm("$(DIR_OUT)/HillCoeff_w_seq_O.txt")
elseif DATA_TYPE == "IIP"
    readdlm("$(DIR_OUT)/IIP_w_seq_O.txt")
end

# Optionally subtract per-antibody mean (center each row)
if flag_subtract_mean
    mean_IC50_to_subtract = [mean(IC50_single_w_seq[i, .!isnan.(IC50_single_w_seq[i, :])]) for i in 1:length(IC50_single_w_seq[:, 1])]
    for i in 1:length(IC50_single_w_seq[:, 1])
        IC50_single_w_seq[i, :] .-= mean_IC50_to_subtract[i]
    end
end;

In [9]:
# Load antibody and virus identifiers
antibody_unique_catnap_single_abs_temp = readdlm("$(DIR_OUT)/antibody_unique_catnap_single_abs_O.txt")
antibody_unique_catnap_single_abs = [join(antibody_unique_catnap_single_abs_temp[i, :]) for i in 1:length(antibody_unique_catnap_single_abs_temp[:, 1])]
virus_unique_catnap_filtered = readdlm("$(DIR_OUT)/virus_unique_catnap_filtered_O.txt")

# Load MSA and PCA-based sequence features
msa_act           = readdlm("$(DIR_OUT)/msa_act_O.txt", Int)
evl               = readdlm("$(DIR_OUT)/evl_O.txt")
evt               = readdlm("$(DIR_OUT)/evt_O.txt")
msa_filtered_temp = readdlm("$(DIR_OUT)/msa_filtered_O.txt", String)
msa_filtered      = [msa_filtered_temp[n, :] for n in 1:size(msa_filtered_temp, 1)]

seq_headder_filtered = readdlm("$(DIR_OUT)/seq_headder_filtered_O.txt")[:, 1]
idx_HXB2 = [!isnothing(match(r"HXB2", string(x))) for x in seq_headder_filtered]
i_HXB2   = findall(idx_HXB2)[1]

idx_nongap = msa_filtered[i_HXB2] .!= "-"  # non-gap positions based on HXB2 reference
q, L = length(Alpha_set_O), count(idx_nongap)
M    = size(msa_filtered, 1) - 1
projected_seq_PCA = readdlm("$(DIR_OUT)/projected_seq_PCA_O.txt")

# Merge alignment-based and alignment-free PCA features
# Features are interleaved (alignment-free, alignment-based, ...) so that
# contributions from both sources are balanced by descending variance order.
projected_seq_PCA_F = readdlm("$(DIR_OUT)/projected_seq_PCA_O_w_features.txt")
n_var_temp, n_var_F_temp = size(projected_seq_PCA, 1), size(projected_seq_PCA_F, 1)
n_min = n_var_F_temp

projected_seq_PCA_merged = zeros(n_var_temp + n_var_F_temp, size(projected_seq_PCA, 2))
for k in 1:n_min
    projected_seq_PCA_merged[(2*(k-1)+1), :] = copy(projected_seq_PCA_F[k, :])
    projected_seq_PCA_merged[(  2 * k  ), :] = copy(projected_seq_PCA[k, :])
end
for k in (n_min+1):n_var_temp
    projected_seq_PCA_merged[k + n_min, :] = copy(projected_seq_PCA[k, :])
end;

## Step 2: Filter and Mask Data

Filter antibodies/viruses with too few measurements, then apply the withholding
scheme selected by `types_of_masking` for cross-validation.


In [10]:
# Build boolean mask for all genuinely observed entries
idx_ic50_masked = copy(string.(IC50_single_w_seq) .!= "NaN")

# Iteratively exclude antibodies/viruses below the minimum observation threshold
(idx_enough_ic50_values_abs, idx_enough_ic50_values_vrs) = get_idx_to_exclude(
    IC50_single_w_seq, num_minimum_num_ic50_each_abs, num_minimum_num_ic50_each_vrs)

writedlm(@sprintf("%s/idx_enough_ic50_values_abs-%s.txt", DIR_OUT, file_key), idx_enough_ic50_values_abs)
writedlm(@sprintf("%s/idx_enough_ic50_values_vrs-%s.txt", DIR_OUT, file_key), idx_enough_ic50_values_vrs)

# Subset data to retained antibodies and viruses
antibody_in_process      = copy(antibody_unique_catnap_single_abs[idx_enough_ic50_values_abs])
virus_in_process         = copy(virus_unique_catnap_filtered[idx_enough_ic50_values_vrs])
IC50_in_true             = Matrix{Any}(copy(IC50_single_w_seq[idx_enough_ic50_values_abs, idx_enough_ic50_values_vrs]))
idx_ic50_masked          = copy(idx_ic50_masked[idx_enough_ic50_values_abs, idx_enough_ic50_values_vrs])
IC50_in_process          = Matrix{Any}(copy(IC50_in_true))
projected_seq_in_process = copy(projected_seq_PCA_merged)[:, idx_enough_ic50_values_vrs]
N_A, M = size(IC50_in_process)

# Parameters used by biased masking modes (types_of_masking 2/3)
x_num_abs     = Int(floor(0.5 * M))
x_num_abs_min = 10
x_num_abs_max = x_num_abs - 10

# Apply the withholding scheme (modifies IC50_in_process in-place)
idx_ic50_masked_training = get_masking_idx(
    idx_ic50_masked, IC50_in_process, types_of_masking, ratio_withold,
    x_num_abs, x_num_abs_min, x_num_abs_max, projected_seq_in_process, M)
IC50_in_process[.!idx_ic50_masked_training] .= "NaN"

# Save O/N/W status matrix:
#   O = observed (in training set)
#   N = never measured
#   W = withheld (observed but held out for evaluation)
bool_NOW_mat = Matrix{String}(undef, size(idx_ic50_masked_training))
bool_NOW_mat[idx_ic50_masked .== 1]                                        .= "O"
bool_NOW_mat[idx_ic50_masked .!= 1]                                        .= "N"
bool_NOW_mat[(idx_ic50_masked .== 1) .&& (idx_ic50_masked_training .== 0)] .= "W"

writedlm(@sprintf("%s/bool_mat_NaN-Observed-Witheld-%s_%s.txt", DIR_OUT, file_key, DATA_TYPE), bool_NOW_mat)
writedlm(@sprintf("%s/Imputed_mat_training-%s_%s.txt",          DIR_OUT, file_key, DATA_TYPE), IC50_in_true)
writedlm(@sprintf("%s/antibody_in_process-%s_%s.txt",           DIR_OUT, file_key, DATA_TYPE), antibody_in_process)
writedlm(@sprintf("%s/virus_in_process-%s_%s.txt",              DIR_OUT, file_key, DATA_TYPE), virus_in_process);

Update abs = 0, Update vrs = 0


## Export Test-Data Files

Write `used-for-training.{accession,fasta}` and `not-used-for-training.{accession,fasta}`
to `../test-data/` based on the IC50-threshold filter applied above.

- **used-for-training**: viruses with ≥ `num_minimum_num_ic50_each_vrs` measurements
  (included in the GLR training matrix).
- **not-used-for-training**: viruses in `virus_unique_catnap_filtered_O.txt` that were
  excluded by the filter (too few IC50 values — "new" to the model).

These files are consumed by `predict_neutralization_for_new_sequences.ipynb`.

In [11]:
# ── Export used-for-training / not-used-for-training test-data files ──────────
DIR_TEST = "../test-data"
mkpath(DIR_TEST)

# Parse accession IDs from MSA headers (format: CLADE.COUNTRY.YEAR.ACCESSION.GENBANKID)
accession_in_msa_filtered = [split(string(h), ".")[end-1] for h in seq_headder_filtered]
acc2msa_idx = Dict(a => i for (i, a) in enumerate(accession_in_msa_filtered))

# Build virus ID lists from the CATNAP-ordered virus array
virus_ids    = [string(virus_unique_catnap_filtered[i, 1]) for i in 1:size(virus_unique_catnap_filtered, 1)]
used_ids     = virus_ids[ idx_enough_ic50_values_vrs]
not_used_ids = virus_ids[.!idx_enough_ic50_values_vrs]

# HXB2 header and sequence (prepended to every FASTA)
hxb2_header = string(seq_headder_filtered[i_HXB2])
hxb2_seq    = join(msa_filtered[i_HXB2])

for (label, ids) in [("used-for-training", used_ids), ("not-used-for-training", not_used_ids)]
    writedlm("$(DIR_TEST)/$(label).accession", ids)
    open("$(DIR_TEST)/$(label).fasta", "w") do io
        println(io, ">$(hxb2_header)")
        println(io, hxb2_seq)
        for id in ids
            haskey(acc2msa_idx, id) || continue
            idx = acc2msa_idx[id]
            println(io, ">$(seq_headder_filtered[idx])")
            println(io, join(msa_filtered[idx]))
        end
    end
    @printf("Wrote %d entries → %s/%s.{accession,fasta}\n", length(ids), DIR_TEST, label)
end

Wrote 1145 entries → ../test-data/used-for-training.{accession,fasta}
Wrote 1079 entries → ../test-data/not-used-for-training.{accession,fasta}


## Step 3: Grouped Linear Regression (GLR) Imputation

**Algorithm overview:**
1. Compute pairwise cosine similarity between antibodies from overlapping IC50 observations.
2. Apply spectral clustering to group antibodies into `k_max` clusters.
3. Fit grouped linear regression: each cluster shares a regularised predictor
   mapping viral PCA features → neutralization value.
   - Single-antibody clusters: independent L2-regularised regression.
   - Multi-antibody clusters: coupled regression with a Laplacian regularisation term
     (`λ2_eff`) that encourages smoothness between similar antibodies.
4. Apply the fitted model to impute all missing entries.


In [12]:
# Build pairwise antibody similarity matrix and cluster with spectral clustering
# (See tools_imputation_minimum.jl: compute_antibody_similarity_matrix, spectral_clustering)
(cosign_abs_similarity, similarity_matrix) = compute_antibody_similarity_matrix(IC50_in_process, N_A, M)
labels = spectral_clustering(similarity_matrix, k_max, n_max_eigen);

Eigenvectors shape: (806, 806)
Data for k-means shape: (51, 806)


In [13]:
# (Optional) Export antibody cluster assignments with their known binding classes
if flag_similarities_out
    (abs_class_set, index_sort_by_abs_type, num_entry_of_class) = get_abs_class(antibody_in_process, fname_abs_class)
    antibody_name_to_class = Dict(zip(antibody_in_process, abs_class_set))
    idx_kmeans, idx_kmeans_temp = [], []
    for i in 1:k_max
        global idx_kmeans, idx_kmeans_temp
        idx_kmeans = vcat(idx_kmeans, collect(1:N_A)[labels .== i])
        [push!(idx_kmeans_temp, i) for _ in 1:count(labels .== i)]
    end
    df = DataFrame(
        abs_id    = antibody_in_process[idx_kmeans],
        abs_class = [antibody_name_to_class[x] for x in antibody_in_process[idx_kmeans]],
        abs_label = idx_kmeans_temp)
    CSV.write(@sprintf("%s/abs_clustering-%s_%s.csv", DIR_OUT, file_key, DATA_TYPE), df)
end

# Sort cluster indices by cluster size (smallest → largest)
set_of_num_each_class     = [count(labels .== i) for i in 1:k_max]
idx_sort_class_number     = sortperm(set_of_num_each_class)
set_of_indices_each_class = [collect(1:N_A)[labels .== i] for i in idx_sort_class_number];

In [14]:
### Fit GLR parameters for each cluster ###
# For each cluster, solve: θ = argmin_θ ||IC50 - X·θ||² + λ1·||θ||² + λ2·θ'·(L⊗I)·θ
# where X = projected_seq_PCA, L = graph Laplacian of the within-cluster similarity matrix.
d_eff_set       = []
θ_set_efficient = []

for i in 1:k_max
    indicies_temp = set_of_indices_each_class[i]

    if length(indicies_temp) == 1
        # --- Single-antibody cluster: standard L2-regularised regression ---
        n_abs = indicies_temp[1]
        idx   = string.(IC50_in_process[n_abs, :]) .!= "NaN"
        d_eff = Int(floor(count(idx) * d_scale))
        d_eff = minimum([d_eff, d_eff_max])
        if d_eff < 2; d_eff = 2; end
        @printf("itr=%d, # of obs = %d, d_eff=%d\n", i, count(idx), d_eff)
        if count(idx) > 0
            X        = projected_seq_in_process[1:d_eff, idx]'
            mat_temp = X' * X + λ1_eff * I
            psi_temp = float.(projected_seq_in_process[1:d_eff, idx]) * float.(IC50_in_process[n_abs, idx])
            θ_temp   = mat_temp \ psi_temp
            push!(θ_set_efficient, copy(θ_temp))
            push!(d_eff_set, d_eff)
        else
            push!(θ_set_efficient, zeros(d_eff))
            push!(d_eff_set, d_eff)
        end

    else
        # --- Multi-antibody cluster: Laplacian-regularised grouped regression ---
        num_abs_in_a_class = length(indicies_temp)
        # d_eff is based on the union of observed viruses across all abs in the cluster
        idx_eff = [false for _ in 1:M]
        for n in 1:num_abs_in_a_class
            idx_eff = idx_eff .|| (string.(IC50_in_process[indicies_temp[n], :]) .!= "NaN")
        end
        d_eff = Int(floor(count(idx_eff) * d_scale))
        d_eff = minimum([d_eff, d_eff_max])
        if d_eff < 2; d_eff = 2; end
        @printf("#_of_abs = %d, itr=%d, d_eff=%d\n", num_abs_in_a_class, i, d_eff)

        if flag_eff_GLR
            # Efficient variant using the Woodbury matrix identity
            Xi     = zeros(num_abs_in_a_class * d_eff, num_abs_in_a_class * d_eff)
            Xi_inv = zeros(num_abs_in_a_class * d_eff, num_abs_in_a_class * d_eff)
            psi    = zeros(num_abs_in_a_class * d_eff)
            for n in 1:num_abs_in_a_class
                n_abs = indicies_temp[n]
                idx   = string.(IC50_in_process[n_abs, :]) .!= "NaN"
                if count(idx) > 0
                    X       = projected_seq_in_process[1:d_eff, idx]'
                    mat_xi  = X' * X + λ1_eff * I
                    Xi[km(n,1,d_eff):km(n,d_eff,d_eff), km(n,1,d_eff):km(n,d_eff,d_eff)]     = copy(mat_xi)
                    Xi_inv[km(n,1,d_eff):km(n,d_eff,d_eff), km(n,1,d_eff):km(n,d_eff,d_eff)] = inv(mat_xi)
                    psi[km(n,1,d_eff):km(n,d_eff,d_eff)] = float.(projected_seq_in_process[1:d_eff, idx]) * float.(IC50_in_process[n_abs, idx])
                end
            end
            idx_to_look = [j ∈ indicies_temp for j in 1:N_A]
            W_min   = cosign_abs_similarity[idx_to_look, idx_to_look]
            Lap_min = compute_laplacian(W_min)
            (evl_L, evt_L) = eigen(Lap_min)
            rank_rho = count(evl_L .> 1e-4)
            rho_set  = zeros(num_abs_in_a_class, rank_rho)
            for r in 1:rank_rho
                rho_set[:, r] = sqrt(evl_L[end+1-r]) * evt_L[:, end+1-r]
            end
            @time θ_temp = get_inv_of_sum_matrix_sparse(Xi_inv, sqrt(λ2_eff) * rho_set) * psi
            push!(θ_set_efficient, copy(θ_temp))
            push!(d_eff_set, d_eff)
        else
            # Standard GLR with Laplacian-Kronecker coupling
            Xi  = zeros(num_abs_in_a_class * d_eff, num_abs_in_a_class * d_eff)
            psi = zeros(num_abs_in_a_class * d_eff)
            for n in 1:num_abs_in_a_class
                n_abs = indicies_temp[n]
                idx   = string.(IC50_in_process[n_abs, :]) .!= "NaN"
                X      = projected_seq_in_process[1:d_eff, idx]'
                mat_xi = X' * X + λ1_eff * I
                Xi[km(n,1,d_eff):km(n,d_eff,d_eff), km(n,1,d_eff):km(n,d_eff,d_eff)] = copy(mat_xi)
                psi[km(n,1,d_eff):km(n,d_eff,d_eff)] = float.(projected_seq_in_process[1:d_eff, idx]) * float.(IC50_in_process[n_abs, idx])
            end
            idx_to_look = [j ∈ indicies_temp for j in 1:N_A]
            W_min   = cosign_abs_similarity[idx_to_look, idx_to_look]
            Lap_min = compute_laplacian(W_min)
            @time θ_temp = (Xi + kron(λ2_eff * Lap_min, diagm(0 => ones(d_eff)))) \ psi
            push!(θ_set_efficient, copy(θ_temp))
            push!(d_eff_set, d_eff)
        end
    end

    # Export GLR parameters if requested
    if flag_param_out_GLR
        vec_θ_csv, site_id_vec_csv, class_vec_csv = [], [], []
        for i_class in 1:minimum([k_max, size(θ_set_efficient, 1)])
            for ii in 1:length(θ_set_efficient[i_class])
                push!(vec_θ_csv,       θ_set_efficient[i_class][ii])
                push!(site_id_vec_csv, ii)
                push!(class_vec_csv,   i_class)
            end
        end
        df = DataFrame(class_id = class_vec_csv, site_id = site_id_vec_csv, param = vec_θ_csv)
        CSV.write(@sprintf("%s/GLR_parameter-%s_%s.csv", DIR_OUT, file_key, DATA_TYPE), df)
    end
end

itr=1, # of obs = 17, d_eff=8
itr=2, # of obs = 18, d_eff=9
itr=3, # of obs = 30, d_eff=15
itr=4, # of obs = 10, d_eff=5
itr=5, # of obs = 37, d_eff=18
itr=6, # of obs = 12, d_eff=6
itr=7, # of obs = 10, d_eff=5
itr=8, # of obs = 11, d_eff=5
itr=9, # of obs = 10, d_eff=5
itr=10, # of obs = 20, d_eff=10
itr=11, # of obs = 120, d_eff=60
itr=12, # of obs = 84, d_eff=42
itr=13, # of obs = 12, d_eff=6
itr=14, # of obs = 12, d_eff=6
itr=15, # of obs = 12, d_eff=6
itr=16, # of obs = 20, d_eff=10
itr=17, # of obs = 12, d_eff=6
itr=18, # of obs = 12, d_eff=6
itr=19, # of obs = 30, d_eff=15
itr=20, # of obs = 33, d_eff=16
itr=21, # of obs = 19, d_eff=9
itr=22, # of obs = 25, d_eff=12
itr=23, # of obs = 12, d_eff=6
itr=24, # of obs = 49, d_eff=24
itr=25, # of obs = 27, d_eff=13
itr=26, # of obs = 39, d_eff=19
itr=27, # of obs = 16, d_eff=8
itr=28, # of obs = 8, d_eff=4
itr=29, # of obs = 7, d_eff=3
itr=30, # of obs = 13, d_eff=6
itr=31, # of obs = 37, d_eff=18
itr=32, # of obs = 25, d_eff=12
itr=

In [15]:
# Map each antibody ID to its (cluster_index, within_cluster_order)
# then apply the fitted θ to impute all missing IC50 values
absID2class_orderID = build_absID2class_orderID(set_of_indices_each_class, k_max)
ic50_imputed_GLR    = apply_glr_imputation(
    IC50_in_process, projected_seq_in_process,
    θ_set_efficient, d_eff_set, absID2class_orderID, N_A)

# Evaluate on withheld entries
idx_only_withholded  = idx_ic50_masked .&& .!idx_ic50_masked_training
true_ic50_vec        = IC50_in_true[idx_only_withholded]
imputed_GLR_ic50_vec = ic50_imputed_GLR[idx_only_withholded]
R_Pearson_GLR  = cor(true_ic50_vec, imputed_GLR_ic50_vec)
R_Spearman_GLR = corspearman(float.(true_ic50_vec), float.(imputed_GLR_ic50_vec))
@printf("GLR — Pearson r = %.3f, Spearman ρ = %.3f\n", R_Pearson_GLR, R_Spearman_GLR);

GLR — Pearson r = 0.750, Spearman ρ = 0.751


## Step 4: Save Outputs

Export the full imputed matrix, withheld-value comparisons, and the rank-dependency analysis.


In [16]:
# Export withheld entries with true vs. imputed IC50 values
if flag_out_imputation
    mat_withold = (idx_ic50_masked_training .== 0) .&& (idx_ic50_masked .== 1)
    vec_ic50_csv, vec_withold, vec_abs_csv, vec_vrs_csv, vec_ic50_glr_csv = [], [], [], [], []
    for i_abs in 1:N_A, i_vrs in 1:M
        if mat_withold[i_abs, i_vrs]
            push!(vec_ic50_csv,     IC50_in_true[i_abs, i_vrs])
            push!(vec_withold,      mat_withold[i_abs, i_vrs])
            push!(vec_abs_csv,      antibody_in_process[i_abs])
            push!(vec_vrs_csv,      virus_in_process[i_vrs])
            push!(vec_ic50_glr_csv, ic50_imputed_GLR[i_abs, i_vrs])
        end
    end
    df = DataFrame(
        abs_name         = vec_abs_csv,
        vrs_name         = vec_vrs_csv,
        true_ic50        = vec_ic50_csv,
        withold          = vec_withold,
        imputed_ic50_GLR = vec_ic50_glr_csv)
    CSV.write(@sprintf("%s/imputed_ic50-%s_%s.csv", DIR_OUT, file_key, DATA_TYPE), df)
end

# Compute SVD-based low-rank approximation (starting with rank_set[1])
rank_take = rank_set[1]
(L_GLR_svd_temp, σ_GLR, U_GLR, V_GLR, min_ic50_GLR) = making_lowrank_mat_return_UV(ic50_imputed_GLR, rank_take)
writedlm(@sprintf("%s/Imputed_mat_GLR-%s_%s.txt", DIR_OUT, file_key, DATA_TYPE), ic50_imputed_GLR)

# Identify the optimal rank (≥95% cumulative variance)
cumsum_GLR = cumsum(σ_GLR .^ 2) ./ sum(σ_GLR .^ 2)
i_opt_GLR  = collect(1:length(cumsum_GLR))[cumsum_GLR .>= 0.95][1]

R_Pearson_GLR_temp  = cor(true_ic50_vec, L_GLR_svd_temp[idx_only_withholded])
R_Spearman_GLR_temp = corspearman(float.(true_ic50_vec), float.(L_GLR_svd_temp[idx_only_withholded]));

In [17]:
# Export rank vs. accuracy table
fout = open(@sprintf("%s/rank_dependency_id-%s_%s.txt", DIR_OUT, file_key, DATA_TYPE), "w")
println(fout, "rank Pearson_GLR Spearman_GLR")
println(fout, @sprintf("%d %.3e %.3e", rank_take, R_Pearson_GLR_temp, R_Spearman_GLR_temp))
for rank_take in rank_set[2:end]
    L_GLR_svd_temp      = making_lowrank_mat_given_UV(σ_GLR, rank_take, U_GLR, V_GLR, min_ic50_GLR)
    R_Pearson_GLR_temp  = cor(true_ic50_vec, L_GLR_svd_temp[idx_only_withholded])
    R_Spearman_GLR_temp = corspearman(float.(true_ic50_vec), float.(L_GLR_svd_temp[idx_only_withholded]))
    println(fout, @sprintf("%d %.3e %.3e", rank_take, R_Pearson_GLR_temp, R_Spearman_GLR_temp))
end
println(fout, @sprintf("Full %.3e %.3e", R_Pearson_GLR, R_Spearman_GLR))
close(fout)

# Export accuracy at the optimal rank
L_GLR_svd_temp      = making_lowrank_mat_given_UV(σ_GLR, i_opt_GLR, U_GLR, V_GLR, min_ic50_GLR)
R_Pearson_GLR_temp  = cor(true_ic50_vec, L_GLR_svd_temp[idx_only_withholded])
R_Spearman_GLR_temp = corspearman(float.(true_ic50_vec), float.(L_GLR_svd_temp[idx_only_withholded]))
fout = open(@sprintf("%s/rank_dependency_optimal_id-%s_%s.txt", DIR_OUT, file_key, DATA_TYPE), "w")
println(fout, "rank Method Pearson Spearman")
println(fout, @sprintf("%d GLR %.3e %.3e", i_opt_GLR, R_Pearson_GLR_temp, R_Spearman_GLR_temp))
close(fout);